# 06 — Preprocesamiento y splits

Notebook **fundacional** del bloque de modelado: discretiza el *target*, decide la estrategia de validacion, aplica el *pipeline* de preprocesamiento y guarda los *splits* en `data/processed/` para que los notebooks 07-09 los consuman directamente.

## Decisiones tomadas en este notebook

| Decision | Valor | Justificacion |
|----------|-------|---------------|
| **Target** | `crimen_violento_total` discretizado en 4 clases | Definido en `README.md` |
| **Discretizacion** | bajo=0, medio=1-5, alto=6-20, critico>20 | Rangos del *Planteamiento del problema* |
| **Validacion HP** | `TimeSeriesSplit(n_splits=5)` sobre train | Hay temporalidad — K-fold *random* contamina con futuro |
| **Holdout final** | 2025 completo (12 meses x 72 munis = 864 filas) | Evaluacion no sesgada del modelo final |
| **Train/Val rapido** | train: 2016-2023, val: 2024 | Para iteracion rapida sin correr CV completa |
| **Scaling** | `StandardScaler` sobre continuas, *passthrough* sobre binarias/ordinales | LR/SVM necesitan escala; arboles son invariantes |
| **Identificadores** | `Cve. Municipio` y `fecha` se preservan pero NO se usan como *features* | `Cve. Municipio` se podria usar como *group* en `GroupKFold` mas adelante |

## Artefactos generados

```
data/processed/
  X_train.parquet, X_val.parquet, X_holdout.parquet      <- features escaladas
  y_train.parquet, y_val.parquet, y_holdout.parquet      <- target multiclase (0-3)
  splits_meta.parquet                                     <- indices, fechas, municipio por fila
models/
  preprocessor.joblib                                     <- ColumnTransformer fitteado en train
  target_bins.json                                        <- definicion de las clases para inferencia
```

Todos se versionan con DVC (`dvc add data/processed/*.parquet models/preprocessor.joblib`) y la corrida se registra en MLflow (experimento `preprocessing`).

## 1. Imports y configuracion

In [1]:
import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

from sonalert.config import MODELS_DIR, PROCESSED_DATA_DIR

RANDOM_STATE = 42

MODELS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

2026-05-19 11:44:28.047 | INFO     | sonalert.config:<module>:11 - PROJ_ROOT path is: /lustre/home/estudiante_57/sonAlert


In [2]:
# MLflow opcional: si no hay credenciales todavia, el notebook corre y guarda artifacts
# pero no logea a DagsHub. Cuando termines el bootstrap (ver references/MLOPS_SETUP.md)
# vuelve a correrlo y los runs apareceran en https://dagshub.com/<owner>/sonAlert/experiments

MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI")
MLFLOW_ACTIVE = bool(MLFLOW_URI)

if MLFLOW_ACTIVE:
    import mlflow

    mlflow.set_tracking_uri(MLFLOW_URI)
    mlflow.set_experiment("preprocessing")
    print(f"[MLflow] activo, tracking a: {MLFLOW_URI}")
else:
    print("[MLflow] inactivo (falta MLFLOW_TRACKING_URI en .env). El notebook corre igual y guarda los parquet/joblib.")

[MLflow] activo, tracking a: https://dagshub.com/RovleZMex/sonAlert.mlflow


## 2. Cargar el panel de *features*

In [3]:
panel = pd.read_parquet(PROCESSED_DATA_DIR / "panel_features.parquet")
panel["fecha"] = pd.to_datetime(panel["fecha"])
panel = panel.sort_values(["fecha", "Cve. Municipio"]).reset_index(drop=True)

print(f"panel_features.parquet: {panel.shape}")
print(f"  Rango fechas:    {panel['fecha'].min().date()} -> {panel['fecha'].max().date()}")
print(f"  Municipios:      {panel['Cve. Municipio'].nunique()}")
print(f"  Columnas totales:{panel.shape[1]}")
print(f"  NaN totales:     {panel.isna().sum().sum()}")
panel.head()

panel_features.parquet: (8640, 155)
  Rango fechas:    2016-01-01 -> 2025-12-01
  Municipios:      72
  Columnas totales:155
  NaN totales:     0


,Cve. Municipio,fecha,extorsion,feminicidio,homicidio_culposo,homicidio_doloso,lesiones,otros,robo_sin_violencia,robo_violento,...,otros_lag3,otros_lag12,otros_roll3_mean,otros_roll6_mean,otros_roll12_mean,otros_roll3_std,otros_yoy_change,covid_lockdown,covid_period,crimen_violento_total
0,26001,2016-01-01,0.0,0.0,0.0,0.0,0.0,5.0,1.0,8.0,...,0.0,0.0,0.666667,0.500000,0.333333,1.154701,5.000000,0,0,8.0
1,26002,2016-01-01,0.0,0.0,1.0,4.0,3.0,12.0,9.0,4.0,...,41.0,32.0,28.666667,30.500000,32.666667,13.650397,-0.606061,0,0,8.0
2,26003,2016-01-01,0.0,0.0,0.0,0.0,2.0,5.0,3.0,0.0,...,9.0,6.0,7.333333,6.000000,7.166667,1.527525,-0.142857,0,0,0.0
3,26004,2016-01-01,0.0,0.0,3.0,0.0,1.0,2.0,1.0,1.0,...,7.0,1.0,3.666667,3.666667,3.916667,2.886751,0.500000,0,0,1.0
4,26005,2016-01-01,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,1.000000,0,0,2.0


## 3. Discretizacion del *target*

`crimen_violento_total` (suma de `homicidio_doloso + feminicidio + robo_violento + secuestro`) se discretiza en 4 clases segun los rangos definidos en el `README.md`:

| Clase | Codigo | Rango |
|-------|--------|-------|
| bajo | 0 | exactamente 0 |
| medio | 1 | 1 - 5 |
| alto | 2 | 6 - 20 |
| critico | 3 | > 20 |

In [4]:
TARGET_BINS = {
    "variable_continua": "crimen_violento_total",
    "clases": {"bajo": 0, "medio": 1, "alto": 2, "critico": 3},
    "cortes": [-1, 0, 5, 20, np.inf],  # bins de pd.cut, left-exclusive right-inclusive
    "labels": [0, 1, 2, 3],
}


def discretize(series: pd.Series) -> pd.Series:
    return pd.cut(series, bins=TARGET_BINS["cortes"], labels=TARGET_BINS["labels"]).astype(int)


panel["y"] = discretize(panel["crimen_violento_total"])

print("Distribucion global del target:")
print(panel["y"].value_counts().sort_index().rename({0: "bajo", 1: "medio", 2: "alto", 3: "critico"}))
print(f"\nPorcentaje critico: {(panel['y'] == 3).mean():.2%}")

Distribucion global del target:
y
bajo       5586
medio      1931
alto        589
critico     534
Name: count, dtype: int64

Porcentaje critico: 6.18%


## 4. Estrategia de validacion

**Decision:** holdout temporal + `TimeSeriesSplit` para HP search.

**Por que NO K-fold *random*:** las *features* incluyen lags y *rolling* hasta de 12 meses; un fold random pondria datos del futuro en el set de entrenamiento, inflando metricas y haciendo el modelo inutil en produccion.

**Por que NO un solo split:** un solo punto de evaluacion sobre 12 meses es ruidoso. `TimeSeriesSplit(5)` da 5 ventanas crecientes con val tambien de ~12 meses cada una.

**Splits:**
- `holdout` = 2025 (864 filas) — **no se toca hasta el final del proyecto**
- `train` + `val` = 2016-2024 (7776 filas) — se usa con `TimeSeriesSplit(5)` durante HP search
- Vista rapida: `train_fast` = 2016-2023 (6048 filas), `val_fast` = 2024 (864 filas)

In [5]:
HOLDOUT_FROM = pd.Timestamp("2025-01-01")
VAL_FAST_FROM = pd.Timestamp("2024-01-01")

mask_holdout = panel["fecha"] >= HOLDOUT_FROM
mask_val = (panel["fecha"] >= VAL_FAST_FROM) & (panel["fecha"] < HOLDOUT_FROM)
mask_train = panel["fecha"] < VAL_FAST_FROM

print(f"train:   {mask_train.sum():>5} filas  ({panel.loc[mask_train, 'fecha'].min().date()} -> {panel.loc[mask_train, 'fecha'].max().date()})")
print(f"val:     {mask_val.sum():>5} filas  ({panel.loc[mask_val, 'fecha'].min().date()} -> {panel.loc[mask_val, 'fecha'].max().date()})")
print(f"holdout: {mask_holdout.sum():>5} filas  ({panel.loc[mask_holdout, 'fecha'].min().date()} -> {panel.loc[mask_holdout, 'fecha'].max().date()})")

print("\nDistribucion del target por split (% de cada clase):")
dist = pd.DataFrame({
    "train":   panel.loc[mask_train, "y"].value_counts(normalize=True).sort_index(),
    "val":     panel.loc[mask_val, "y"].value_counts(normalize=True).sort_index(),
    "holdout": panel.loc[mask_holdout, "y"].value_counts(normalize=True).sort_index(),
}).rename(index={0: "bajo", 1: "medio", 2: "alto", 3: "critico"})
dist.style.format("{:.2%}")

train:    6912 filas  (2016-01-01 -> 2023-12-01)
val:       864 filas  (2024-01-01 -> 2024-12-01)
holdout:   864 filas  (2025-01-01 -> 2025-12-01)

Distribucion del target por split (% de cada clase):


,train,val,holdout
y,,,
bajo,64.31%,64.93%,67.13%
medio,22.19%,23.26%,22.69%
alto,7.00%,6.13%,6.02%
critico,6.50%,5.67%,4.17%


## 5. Seleccion de *features*

Excluir del conjunto X:
- **Identificadores:** `Cve. Municipio`, `fecha` (se preservan en `splits_meta.parquet` para *post-hoc* analysis y `GroupKFold`).
- **Conteos del mes actual** (`homicidio_doloso`, `feminicidio`, `robo_violento`, `secuestro`, etc.): NO se pueden usar como X — son parte del *target* o serian *leakage* trivial. Los lags y rolling de esos mismos counts SI son validos (estan *shifted* en `features.py`).
- **`crimen_violento_total`** (la variable continua) y la columna `y` derivada.

Conservamos como X: marginacion, demografia, tasas per capita historicas (calculadas sobre lags), Zonas Salva, temporal cyclic, lags, rolling means/std, YoY, COVID dummies, `pop_bucket`.

In [6]:
CRUDOS = [
    "homicidio_doloso", "homicidio_culposo", "feminicidio", "lesiones",
    "robo_violento", "robo_sin_violencia", "violacion_sexual", "secuestro",
    "extorsion", "violencia_familiar", "otros",
]
EXCLUIR_DE_X = ["Cve. Municipio", "fecha", "crimen_violento_total", "y"] + CRUDOS

feature_cols = [c for c in panel.columns if c not in EXCLUIR_DE_X]
print(f"Features finales: {len(feature_cols)}")
print(f"Excluidas: {len(EXCLUIR_DE_X)} ({EXCLUIR_DE_X})")

Features finales: 141
Excluidas: 15 (['Cve. Municipio', 'fecha', 'crimen_violento_total', 'y', 'homicidio_doloso', 'homicidio_culposo', 'feminicidio', 'lesiones', 'robo_violento', 'robo_sin_violencia', 'violacion_sexual', 'secuestro', 'extorsion', 'violencia_familiar', 'otros'])


## 6. Pipeline de preprocesamiento

`ColumnTransformer`:
- `StandardScaler` para todas las *features* continuas.
- *Passthrough* para las binarias y ordinales: `covid_lockdown`, `covid_period`, `pop_bucket`, `mes_sin`, `mes_cos` (ya estan en escala util).

El transformer se ajusta **solo sobre train** (no train+val) para evitar *leakage* hacia el set de validacion.

In [7]:
PASSTHROUGH = [c for c in ["covid_lockdown", "covid_period", "pop_bucket", "mes_sin", "mes_cos"] if c in feature_cols]
ESCALAR = [c for c in feature_cols if c not in PASSTHROUGH]

preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), ESCALAR),
        ("passthrough", "passthrough", PASSTHROUGH),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
preprocessor.set_output(transform="pandas")

print(f"Escaladas ({len(ESCALAR)}):    {ESCALAR[:5]}... (+{len(ESCALAR)-5} mas)")
print(f"Passthrough ({len(PASSTHROUGH)}): {PASSTHROUGH}")

Escaladas (136):    ['POB_TOT', 'ANALF', 'SBASC', 'OVSDE', 'OVSEE']... (+131 mas)
Passthrough (5): ['covid_lockdown', 'covid_period', 'pop_bucket', 'mes_sin', 'mes_cos']


In [8]:
X_train = preprocessor.fit_transform(panel.loc[mask_train, feature_cols])
X_val = preprocessor.transform(panel.loc[mask_val, feature_cols])
X_holdout = preprocessor.transform(panel.loc[mask_holdout, feature_cols])

y_train = panel.loc[mask_train, "y"].reset_index(drop=True)
y_val = panel.loc[mask_val, "y"].reset_index(drop=True)
y_holdout = panel.loc[mask_holdout, "y"].reset_index(drop=True)

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_holdout = X_holdout.reset_index(drop=True)

print(f"X_train:   {X_train.shape}")
print(f"X_val:     {X_val.shape}")
print(f"X_holdout: {X_holdout.shape}")
print(f"\nMedia post-scaling en train (deberia ser ~0): {X_train[ESCALAR].mean().mean():.6f}")
print(f"Std post-scaling en train (deberia ser ~1):    {X_train[ESCALAR].std().mean():.6f}")

X_train:   (6912, 141)
X_val:     (864, 141)
X_holdout: (864, 141)

Media post-scaling en train (deberia ser ~0): -0.000000
Std post-scaling en train (deberia ser ~1):    1.000072


## 7. Validacion del *split* con `TimeSeriesSplit`

Confirmamos que `TimeSeriesSplit(5)` aplicado sobre train+val (2016-2024) produce *folds* coherentes: cada *fold* entrena con pasado y valida con futuro inmediato.

In [9]:
# Concatenamos train+val para mostrar como TimeSeriesSplit los partiria durante HP search
trainval_idx = panel.loc[mask_train | mask_val].sort_values("fecha").index
fechas_trainval = panel.loc[trainval_idx, "fecha"].reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=5)
for i, (tr_idx, va_idx) in enumerate(tscv.split(fechas_trainval)):
    print(
        f"Fold {i + 1}: "
        f"train {fechas_trainval.iloc[tr_idx].min().date()} -> {fechas_trainval.iloc[tr_idx].max().date()} "
        f"({len(tr_idx)} filas) | "
        f"val {fechas_trainval.iloc[va_idx].min().date()} -> {fechas_trainval.iloc[va_idx].max().date()} "
        f"({len(va_idx)} filas)"
    )

Fold 1: train 2016-01-01 -> 2017-06-01 (1296 filas) | val 2017-07-01 -> 2018-12-01 (1296 filas)
Fold 2: train 2016-01-01 -> 2018-12-01 (2592 filas) | val 2019-01-01 -> 2020-06-01 (1296 filas)
Fold 3: train 2016-01-01 -> 2020-06-01 (3888 filas) | val 2020-07-01 -> 2021-12-01 (1296 filas)
Fold 4: train 2016-01-01 -> 2021-12-01 (5184 filas) | val 2022-01-01 -> 2023-06-01 (1296 filas)
Fold 5: train 2016-01-01 -> 2023-06-01 (6480 filas) | val 2023-07-01 -> 2024-12-01 (1296 filas)


## 8. Guardar artefactos

In [10]:
from sonalert.config import PROJ_ROOT

X_train.to_parquet(PROCESSED_DATA_DIR / "X_train.parquet")
X_val.to_parquet(PROCESSED_DATA_DIR / "X_val.parquet")
X_holdout.to_parquet(PROCESSED_DATA_DIR / "X_holdout.parquet")

y_train.to_frame(name="y").to_parquet(PROCESSED_DATA_DIR / "y_train.parquet")
y_val.to_frame(name="y").to_parquet(PROCESSED_DATA_DIR / "y_val.parquet")
y_holdout.to_frame(name="y").to_parquet(PROCESSED_DATA_DIR / "y_holdout.parquet")

splits_meta = panel[["Cve. Municipio", "fecha"]].copy()
splits_meta["split"] = np.where(mask_holdout, "holdout", np.where(mask_val, "val", "train"))
splits_meta.to_parquet(PROCESSED_DATA_DIR / "splits_meta.parquet")

joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

with open(MODELS_DIR / "target_bins.json", "w") as fh:
    json.dump({**TARGET_BINS, "cortes": [float(c) if np.isfinite(c) else "inf" for c in TARGET_BINS["cortes"]]}, fh, indent=2)

print("Artefactos guardados:")
for p in sorted(PROCESSED_DATA_DIR.glob("*.parquet")):
    print(f"  {p.relative_to(PROJ_ROOT)} ({p.stat().st_size / 1024:.1f} KB)")
for p in sorted(MODELS_DIR.glob("*")):
    print(f"  {p.relative_to(PROJ_ROOT)} ({p.stat().st_size / 1024:.1f} KB)")

Artefactos guardados:
  data/processed/X_holdout.parquet (205.7 KB)
  data/processed/X_train.parquet (753.4 KB)
  data/processed/X_val.parquet (207.5 KB)
  data/processed/panel_features.parquet (736.1 KB)
  data/processed/splits_meta.parquet (4.8 KB)
  data/processed/y_holdout.parquet (1.8 KB)
  data/processed/y_train.parquet (3.3 KB)
  data/processed/y_val.parquet (1.8 KB)
  models/.gitignore (0.0 KB)
  models/.gitkeep (0.0 KB)
  models/logreg_best.joblib.dvc (0.1 KB)
  models/preprocessor.joblib (14.2 KB)
  models/preprocessor.joblib.dvc (0.1 KB)
  models/target_bins.json (0.2 KB)


## 9. Registrar la corrida en MLflow

In [11]:
if MLFLOW_ACTIVE:
    with mlflow.start_run(run_name="preprocessing_v1") as run:
        mlflow.set_tags({
            "stage": "preprocessing",
            "target": "crimen_violento_total",
            "target_type": "multiclass_4",
            "validation": "TimeSeriesSplit_5+holdout_2025",
        })
        mlflow.log_params({
            "n_features": len(feature_cols),
            "n_scaled": len(ESCALAR),
            "n_passthrough": len(PASSTHROUGH),
            "train_rows": int(mask_train.sum()),
            "val_rows": int(mask_val.sum()),
            "holdout_rows": int(mask_holdout.sum()),
            "holdout_from": str(HOLDOUT_FROM.date()),
            "val_from": str(VAL_FAST_FROM.date()),
        })
        # Distribucion de clases por split como metricas (para tener un baseline visible)
        for split_name, mask in [("train", mask_train), ("val", mask_val), ("holdout", mask_holdout)]:
            for clase, code in TARGET_BINS["clases"].items():
                pct = (panel.loc[mask, "y"] == code).mean()
                mlflow.log_metric(f"class_pct_{split_name}_{clase}", float(pct))

        mlflow.log_artifact(str(MODELS_DIR / "preprocessor.joblib"))
        mlflow.log_artifact(str(MODELS_DIR / "target_bins.json"))
        print(f"[MLflow] run registrado: {run.info.run_id}")
else:
    print("[MLflow] inactivo — corrida no registrada. Completa references/MLOPS_SETUP.md y reejecuta esta celda.")

[MLflow] run registrado: 5b772cc8c7e346f4b634e93d8b534c97
🏃 View run preprocessing_v1 at: https://dagshub.com/RovleZMex/sonAlert.mlflow/#/experiments/0/runs/5b772cc8c7e346f4b634e93d8b534c97
🧪 View experiment at: https://dagshub.com/RovleZMex/sonAlert.mlflow/#/experiments/0


## 10. Proximos pasos

1. **Versionar con DVC** (desde la terminal):
    ```bash
    dvc add data/processed/X_train.parquet data/processed/X_val.parquet data/processed/X_holdout.parquet
    dvc add data/processed/y_train.parquet data/processed/y_val.parquet data/processed/y_holdout.parquet
    dvc add data/processed/splits_meta.parquet models/preprocessor.joblib
    git add data/processed/*.dvc models/preprocessor.joblib.dvc models/target_bins.json
    git commit -m "data: splits preprocesados v1"
    dvc push
    ```
2. **Notebook 07** (baseline logistic) — leer estos parquets y entrenar.
3. **Notebooks 08/09** (companeros) — mismo punto de partida.